RQ3 microsaccade follow-up analyses
======================================
Builds on RQ1_microsaccades.py.  Three things in one place:

  (a) 2 (Group: Artist / Control) x 5 (Level: 1..5) mixed ANOVA on
      microsaccade rate (Hz) and on mean microsaccade amplitude (deg).
      Mixed model:  metric ~ Group * C(Level) + (1 | pid).
      Wald F-tests on the fixed effects give an ANOVA-style table.

  (b) Per-level Welch t-tests comparing Artist vs Control on each of
      the 5 luminance levels — for both rate and amplitude.

  (c) Polar rose of the *direction* of the detected microsaccades,
      Artist vs Control overlaid.  Lets us check whether microsaccade
      direction has the same horizontal-dominant signature seen in the
      RQ3 (full saccade) data.


Outputs to /Users/matyldakornacka/Desktop/DATA_DISS/RQ1/microsaccades/
  RQ1ms_anova_rate.csv
  RQ1ms_anova_amplitude.csv
  RQ1ms_per_level_welch.csv
  RQ1ms_microsaccade_events.csv
  RQ1ms_fig3_polar_rose.png/.pdf

The mixed model was:

metric ~ Group × Level + (1 | pid)

Where:

metric = the dependent variable (total rate, horizontal rate, vertical rate, or amplitude)
Group = fixed effect with 2 levels (Artist vs. Control)
Level = fixed effect with 5 levels (luminance levels 1–5)
Group × Level = fixed interaction effect
(1 | pid) = random intercept by participant (allows each participant to have their own baseline, accounting for repeated measures across levels)
The model was fit using restricted maximum likelihood (LBFGS method in statsmodels), and inference on the fixed effects was conducted via Wald F-tests.

In [1]:

import os, sys, glob, importlib.util, warnings
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# -- import the detector (and constants) from RQ1_microsaccades.py
HERE = '/Users/matyldakornacka/Desktop/DATA_DISS/RQ1'
spec = importlib.util.spec_from_file_location(
    'rq1_ms', os.path.join(HERE, 'RQ1_microsaccades.py'))
ms = importlib.util.module_from_spec(spec); spec.loader.exec_module(ms)

DATA_DIR = ms.DATA_DIR
OUT_DIR  = ms.OUT_DIR              # writes alongside the original script
os.makedirs(OUT_DIR, exist_ok=True)

ART_COL  = ms.ART_COL
CTRL_COL = ms.CTRL_COL
VERTICAL_CODE = ms.VERTICAL_CODE
SAMPLE_HZ     = ms.SAMPLE_HZ

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'figure.dpi': 110, 'savefig.dpi': 240, 'savefig.bbox': 'tight',
})


In [2]:


# ----------------------------------------------------------------- detect
def build_event_table(recs):
    """One row per detected microsaccade plus a per-trial summary for the
    ANOVA.  Now keeps ALL SymThrsh trials (both stimulus axes) and tags
    each event as Horizontal vs Vertical."""
    ev_rows, tr_rows = [], []
    for r in recs:
        rf = r['ResFile']
        for ti in range(len(rf)):
            stim_axis = ('Vertical' if int(rf[ti, 0]) == VERTICAL_CODE
                         else 'Horizontal')
            level = int(rf[ti, 1])
            tr = r['SymThrsh'].TrialLevelData[ti]
            X, Y = ms.clean_xy(tr.XX, tr.YY)
            events = ms.detect_microsaccades(X, Y, r['dpp'])
            n_samples = len(np.asarray(tr.XX))
            dur_s = n_samples / SAMPLE_HZ if n_samples > 0 else np.nan
            for e in events:
                ev_rows.append(dict(
                    pid=r['pid'], group=r['group'],
                    trial=ti, level=level, stim_axis=stim_axis,
                    amp_deg=e['amp_deg'],
                    pkvel_deg_s=e['pkvel_deg_s'],
                    angle_deg=e['angle_deg'],
                    direction=('Horizontal'
                               if ms.is_horizontal(e['angle_deg'])
                               else 'Vertical')))
            n_h = sum(1 for e in events
                      if ms.is_horizontal(e['angle_deg']))
            n_v = len(events) - n_h
            amps = [e['amp_deg'] for e in events] if events else [np.nan]
            pvs  = [e['pkvel_deg_s'] for e in events] if events else [np.nan]
            tr_rows.append(dict(
                pid=r['pid'], group=r['group'], trial=ti,
                level=level, stim_axis=stim_axis,
                duration_s=dur_s,
                n_ms=len(events), n_ms_h=n_h, n_ms_v=n_v,
                rate_hz   =(len(events) / dur_s) if dur_s else np.nan,
                rate_hz_h =(n_h         / dur_s) if dur_s else np.nan,
                rate_hz_v =(n_v         / dur_s) if dur_s else np.nan,
                mean_amp_deg=float(np.nanmean(amps)),
                mean_pkvel_deg_s=float(np.nanmean(pvs))))
    return pd.DataFrame(ev_rows), pd.DataFrame(tr_rows)


def aggregate_cells(per_trial):
    cell = (per_trial.groupby(['pid', 'group', 'level'])
                     .agg(n_trials=('trial', 'count'),
                          n_ms=('n_ms', 'sum'),
                          n_ms_h=('n_ms_h', 'sum'),
                          n_ms_v=('n_ms_v', 'sum'),
                          dur=('duration_s', 'sum'),
                          mean_amp=('mean_amp_deg', 'mean'))
                     .reset_index())
    cell['rate_hz']   = cell.n_ms   / cell.dur
    cell['rate_hz_h'] = cell.n_ms_h / cell.dur
    cell['rate_hz_v'] = cell.n_ms_v / cell.dur
    return cell


In [3]:


# ----------------------------------------------------------------- ANOVA
def mixedlm_anova(df, dv, out_csv):
    """Fit dv ~ Group * C(Level) with pid as random intercept and
    derive a Wald-F ANOVA table for each fixed-effect term."""
    d = df.dropna(subset=[dv]).copy()
    d['Group'] = d['group'].astype('category')
    d['Level'] = d['level'].astype('category')
    md  = smf.mixedlm(f"{dv} ~ C(Group) * C(Level)", d, groups=d['pid'])
    mdf = md.fit(method='lbfgs')

    # statsmodels names: 'C(Group)[T.Control]', 'C(Level)[T.2]',
    # 'C(Group)[T.Control]:C(Level)[T.2]'
    names = list(mdf.params.index)
    pat = {
        'C(Group)':         lambda n: n.startswith('C(Group)[') and ':' not in n,
        'C(Level)':         lambda n: n.startswith('C(Level)[') and ':' not in n,
        'C(Group):C(Level)':lambda n: ('C(Group)' in n) and (':C(Level)[' in n
                                                              or 'C(Level)[' in n.split(':',1)[-1]),
    }
    rows = []
    for t, fn in pat.items():
        idx = [i for i, n in enumerate(names) if fn(n)]
        if not idx: continue
        R = np.zeros((len(idx), len(mdf.params)))
        for k, i in enumerate(idx): R[k, i] = 1
        try:
            wt = mdf.wald_test(R, use_f=True)
            rows.append(dict(term=t, df_num=wt.df_num, df_den=wt.df_denom,
                             F=float(wt.statistic), p=float(wt.pvalue)))
        except Exception:
            rows.append(dict(term=t, df_num=np.nan, df_den=np.nan,
                             F=np.nan, p=np.nan))
    aov = pd.DataFrame(rows)
    aov.to_csv(out_csv, index=False)
    return mdf, aov


In [4]:


# ----------------------------------------------------------------- per-level
def per_level_welch(cell):
    rows = []
    metrics = ('rate_hz', 'rate_hz_h', 'rate_hz_v', 'mean_amp')
    for lvl in sorted(cell.level.unique()):
        sub = cell[cell.level == lvl]
        a = sub[sub.group == 'Artist']
        c = sub[sub.group == 'Control']
        for metric in metrics:
            t, p = stats.ttest_ind(a[metric], c[metric],
                                   equal_var=False, nan_policy='omit')
            rows.append(dict(level=lvl, metric=metric,
                             n_artist=int(a[metric].notna().sum()),
                             n_control=int(c[metric].notna().sum()),
                             M_artist=float(np.nanmean(a[metric])),
                             SD_artist=float(np.nanstd(a[metric], ddof=1)),
                             M_control=float(np.nanmean(c[metric])),
                             SD_control=float(np.nanstd(c[metric], ddof=1)),
                             t=float(t), p=float(p)))
    return pd.DataFrame(rows)


## Figure 3: Polar Rose of Microsaccade Directions

**Construction:**
The polar rose displays the angular distribution of detected microsaccades in 36 bins (10° each) arranged radially around a circle. Each bin's height represents either the raw count or proportion of microsaccades directed within that angular range. Artists (blue) and controls (red/orange) are overlaid side-by-side for direct comparison. The figure uses cardinal labels (Right, Up, Left, Down) for interpretability. Shaded grey wedges mark the horizontal analysis windows (±45° from right/left, >135° from left), corresponding to the RQ3 full-saccade definition.

**What it shows:**
The rose reveals whether microsaccade direction preferences differ between groups. If microsaccades cluster along vertical sectors (Up/Down), it suggests they align with the symmetry axis during the task. If they accumulate in the horizontal wedges (Right/Left), it indicates the same horizontal-dominant signature seen in larger saccades. Equal distribution across all angles would indicate isotropic eye movements.

This visualization complements the aggregate rate and amplitude analyses by exposing directional biases that might be masked in simple summary statistics.

In [ ]:


# ----------------------------------------------------------------- fig 3
def fig3_polar_rose(events):
    """Polar rose of microsaccade angles, Artist vs Control overlaid."""
    nbins = 36
    edges = np.linspace(-np.pi, np.pi, nbins + 1)
    centres = (edges[:-1] + edges[1:]) / 2
    width   = edges[1] - edges[0]

    fig = plt.figure(figsize=(14, 6.8))
    for i, mode in enumerate(['count', 'proportion']):
        ax = fig.add_subplot(1, 2, i + 1, projection='polar')
        ymax = 0
        for grp, col in [('Artist', ART_COL), ('Control', CTRL_COL)]:
            a = np.deg2rad(events.loc[events.group == grp, 'angle_deg']
                                 .values)
            a = np.arctan2(np.sin(a), np.cos(a))
            H, _ = np.histogram(a, bins=edges)
            n_total = (events.group == grp).sum()
            if mode == 'proportion' and H.sum() > 0:
                H = H / H.sum()
            ax.bar(centres, H, width=width * 0.88, bottom=0.0,
                   align='center', color=col, alpha=0.60,
                   edgecolor=col, linewidth=0.8,
                   label=f'{grp} (n = {int(n_total)})')
            ymax = max(ymax, H.max())

        # Set radial axis properties
        ax.set_theta_zero_location('E')
        ax.set_theta_direction(-1)
        ax.set_rlabel_position(45)
        ax.grid(True, lw=0.5, alpha=0.4)
        
        # Cardinal labels (cleaner positioning)
        ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
        ax.set_xticklabels(['Right', 'Up', 'Left', 'Down'], fontsize=10)
        
        # Subplot title
        ax.set_title(('Count' if mode == 'count' else 'Proportion'),
                     fontsize=11, fontweight='bold', pad=20)
        
        # Shaded horizontal wedges (the RQ3 analysis windows)
        for lo, hi in [(-np.pi / 4, np.pi / 4),
                       (3 * np.pi / 4, np.pi),
                       (-np.pi, -3 * np.pi / 4)]:
            ax.fill_between(np.linspace(lo, hi, 40), 0, ymax * 1.15,
                            color='0.75', alpha=0.15, zorder=-1)
        
        # Legend (left panel only)
        if i == 0:
            ax.legend(loc='upper left', bbox_to_anchor=(0.85, 1.15),
                      frameon=False, fontsize=10)

    fig.suptitle('Microsaccade Direction Distribution (All Trials)',
                 fontsize=13, fontweight='bold', y=0.98)
    fig.text(0.5, 0.02,
             'Shaded wedges: horizontal RQ3 windows (±45° from right/left).  '
             'Vertical sectors show microsaccades along the symmetry axis.',
             ha='center', fontsize=9, style='italic', color='#666')
    fig.subplots_adjust(bottom=0.08, top=0.92, wspace=0.3)
    out = os.path.join(OUT_DIR, 'RQ1ms_fig3_polar_rose')
    fig.savefig(out + '.png', dpi=240); fig.savefig(out + '.pdf')
    plt.close(fig)
    print('  wrote', out + '.png')


def fig3b_polar_rose_symmetry(events):
    """Polar rose of microsaccade angles for VERTICAL-AXIS (symmetry) trials only."""
    # Filter to vertical-axis trials only
    events_vert = events[events.stim_axis == 'Vertical'].copy()
    
    nbins = 36
    edges = np.linspace(-np.pi, np.pi, nbins + 1)
    centres = (edges[:-1] + edges[1:]) / 2
    width   = edges[1] - edges[0]

    fig = plt.figure(figsize=(14, 6.8))
    for i, mode in enumerate(['count', 'proportion']):
        ax = fig.add_subplot(1, 2, i + 1, projection='polar')
        ymax = 0
        for grp, col in [('Artist', ART_COL), ('Control', CTRL_COL)]:
            # Filter events: vertical axis trials only
            subset = events_vert[events_vert.group == grp]
            
            if len(subset) == 0:
                continue
            
            a = np.deg2rad(subset.angle_deg.values)
            a = np.arctan2(np.sin(a), np.cos(a))
            H, _ = np.histogram(a, bins=edges)
            n_total = len(subset)
            
            if mode == 'proportion' and H.sum() > 0:
                H = H / H.sum()
            
            ax.bar(centres, H, width=width * 0.88, bottom=0.0,
                   align='center', color=col, alpha=0.60,
                   edgecolor=col, linewidth=0.8,
                   label=f'{grp} (n = {int(n_total)})')
            ymax = max(ymax, H.max())

        ax.set_theta_zero_location('E')
        ax.set_theta_direction(-1)
        ax.set_rlabel_position(45)
        ax.grid(True, lw=0.5, alpha=0.4)
        ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
        ax.set_xticklabels(['Right', 'Up', 'Left', 'Down'], fontsize=10)
        ax.set_title(('Count' if mode == 'count' else 'Proportion'),
                     fontsize=11, fontweight='bold', pad=20)
        
        for lo, hi in [(-np.pi / 4, np.pi / 4),
                       (3 * np.pi / 4, np.pi),
                       (-np.pi, -3 * np.pi / 4)]:
            ax.fill_between(np.linspace(lo, hi, 40), 0, ymax * 1.15,
                            color='0.75', alpha=0.15, zorder=-1)
        
        if i == 0:
            ax.legend(loc='upper left', bbox_to_anchor=(0.85, 1.15),
                      frameon=False, fontsize=10)

    fig.suptitle('Microsaccade Direction Distribution (Vertical-Axis Trials)',
                 fontsize=13, fontweight='bold', y=0.98)
    fig.text(0.5, 0.02,
             'Symmetry axis stimulus direction only. Shaded wedges: horizontal RQ3 windows.',
             ha='center', fontsize=9, style='italic', color='#666')
    fig.subplots_adjust(bottom=0.08, top=0.92, wspace=0.3)
    out = os.path.join(OUT_DIR, 'RQ1ms_fig3b_polar_rose_symmetry')
    fig.savefig(out + '.png', dpi=240); fig.savefig(out + '.pdf')
    plt.close(fig)
    print('  wrote', out + '.png')


def fmtp(p):
    if pd.isna(p): return 'nan'
    return f'{p:.4f}' if p >= 1e-4 else '<.0001'


In [ ]:


# ----------------------------------------------------------------- main
def run():
    recs = ms.load_all()
    na = sum(r['group'] == 'Artist'  for r in recs)
    nc = sum(r['group'] == 'Control' for r in recs)
    print(f'Loaded {len(recs)} participants ({na} artists, {nc} controls)')
    print(f'Re-detecting microsaccades '
          f'(LAMBDA={ms.LAMBDA}, min_dur={ms.MIN_DURATION_MS}ms, '
          f'amp range=[{ms.AMP_MIN_DEG},{ms.AMP_MAX_DEG}]°)\n')

    events, per_trial = build_event_table(recs)
    cell = aggregate_cells(per_trial)
    events.to_csv(os.path.join(OUT_DIR, 'RQ1ms_microsaccade_events.csv'),
                  index=False)
    print(f'  events: {len(events)}    cells (pid x level): {len(cell)}\n')

    # ---- (a) 2x5 mixed ANOVAs --------------------------------------
    for dv, label, fname in [
            ('rate_hz',   'total rate (Hz)',         'RQ1ms_anova_rate.csv'),
            ('rate_hz_h', 'horizontal rate (Hz)',    'RQ1ms_anova_rate_h.csv'),
            ('rate_hz_v', 'vertical rate (Hz)',      'RQ1ms_anova_rate_v.csv'),
            ('mean_amp',  'amplitude (deg)',         'RQ1ms_anova_amplitude.csv')]:
        print(f'=== Mixed ANOVA — {label} ~ Group x Level (random: pid) ===')
        _, aov = mixedlm_anova(cell, dv, os.path.join(OUT_DIR, fname))
        print(aov.to_string(index=False), '\n')

    # ---- (b) per-level Welch --------------------------------------
    pl = per_level_welch(cell)
    pl.to_csv(os.path.join(OUT_DIR, 'RQ1ms_per_level_welch.csv'), index=False)
    print('\n=== Per-level Welch t-tests (Artist vs Control) ===')
    for _, r in pl.iterrows():
        print(f"  Level {int(r.level)}  {r.metric:<8}  "
              f"A={r.M_artist:+.3f}±{r.SD_artist:.3f} (n={int(r.n_artist)})   "
              f"C={r.M_control:+.3f}±{r.SD_control:.3f} (n={int(r.n_control)})   "
              f"t={r.t:+.2f}  p={fmtp(r.p)}")

    # ---- (c) polar rose -------------------------------------------
    print('\nFig 3  — microsaccade direction polar rose (all trials)')
    fig3_polar_rose(events)
    
    print('\nFig 3b — microsaccade direction polar rose (symmetry trials only)')
    fig3b_polar_rose_symmetry(events)

    # quick descriptive comparison to the RQ3 horizontal-wedge proportion
    a = np.deg2rad(events.loc[events.group == 'Artist',  'angle_deg'].values)
    c = np.deg2rad(events.loc[events.group == 'Control', 'angle_deg'].values)
    def hwedge_pct(arr):
        a2 = np.abs(((np.degrees(arr) + 180) % 360) - 180)
        return float(np.mean((a2 < 45) | (a2 > 135)))
    print(f'\nProportion of microsaccades in the RQ3 horizontal wedges:')
    print(f'  Artists  = {hwedge_pct(a):.3f}')
    print(f'  Controls = {hwedge_pct(c):.3f}')

    print('\nDone. Outputs in:', OUT_DIR)


In [10]:
run()

Loaded 31 participants (13 artists, 18 controls)
Re-detecting microsaccades (LAMBDA=5.0, min_dur=3ms, amp range=[0.03,1.0]°)

  events: 2726    cells (pid x level): 155

=== Mixed ANOVA — total rate (Hz) ~ Group x Level (random: pid) ===
  events: 2726    cells (pid x level): 155

=== Mixed ANOVA — total rate (Hz) ~ Group x Level (random: pid) ===
             term  df_num  df_den        F        p
         C(Group)     1.0     145 0.737852 0.391768
         C(Level)     4.0     145 0.173558 0.951657
C(Group):C(Level)     4.0     145 1.370322 0.247114 

=== Mixed ANOVA — horizontal rate (Hz) ~ Group x Level (random: pid) ===
             term  df_num  df_den        F        p
         C(Group)     1.0     145 0.737852 0.391768
         C(Level)     4.0     145 0.173558 0.951657
C(Group):C(Level)     4.0     145 1.370322 0.247114 

=== Mixed ANOVA — horizontal rate (Hz) ~ Group x Level (random: pid) ===
             term  df_num  df_den        F        p
         C(Group)     1.0     14

A 2 (Group) × 5 (Level) linear mixed-effects model with random intercepts by participant was fitted to total microsaccade rate. Wald F-tests on fixed effects revealed no main effect of group, F(1, 145) = 0.74, p = .392, luminance level, F(4, 145) = 0.17, p = .952, and a non-significant Group × Level interaction, F(4, 145) = 1.37, p = .247. Post-hoc independent-samples t-tests (Welch's correction for unequal variances) at each level showed controls exhibited significantly higher total rates at 48% contrast threshold, t(29) = −2.52, p = .018, and Level 5, t(29) = −2.16, p = .040. For amplitude, the mixed model revealed a significant main effect of level, F(4, 145) = 5.01, p < .001, and a significant Group × Level interaction, F(4, 145) = 3.42, p = .010, driven by elevated amplitude in artists at 96% contrast. 